In [1]:
!pip install -q captum shap lime sentence-transformers pandas matplotlib seaborn


# Cuadernillo Experimental XAI: Validación de Robustez en Muestras Aleatorias del Corpus

**Proyecto:** Tesis - Detección de Redundancia Textual en Español y XAI  
**Modelo:** `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`  
**Objetivo:** Validar la robustez y fidelidad mecanicista de la suite XAI sobre una muestra aleatoria no curada de 10 pares (5 redundantes y 5 no redundantes).


In [2]:
import os, gc, time, math, copy, json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.stats import spearmanr, pearsonr
from sklearn.linear_model import Ridge, LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel

np.random.seed(42)
torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Entorno configurado correctamente. Dispositivo: {device}')


Entorno configurado correctamente. Dispositivo: cpu


## 1. Muestreo Aleatorio Balanceado de 10 Pares del Corpus
Se seleccionan 5 pares redundantes ($s \ge 0.75$) y 5 pares no redundantes ($s \le 0.38$) directamente de `dataset_con_similitudes.csv`.


In [3]:
# Pares aleatorios balanceados extraídos del corpus
RANDOM_PAIRS = [
  {
    "doc_id": 1195,
    "class": "False",
    "type": "Redundante",
    "text_a": "\"Papá está muy preocupado porque el médico le ha dicho que pronto se irá con el abuelo, pero papá me dijo que el abuelo estaba en un sitio maravilloso en el que no sufría\", argumenta el pequeño, incapaz de comprender el ánimo sombrío de su progenitor.",
    "text_b": "Cuando yo estaba triste porque el abuelo ya no estaba con nosotros él me dijo que no tenía que estar triste\", explica el niño, evidenciando las contradicciones del padre.",
    "similarity": 0.7687070369720459
  },
  {
    "doc_id": 131,
    "class": "True",
    "type": "Redundante",
    "text_a": "Principal 13 sentenciados por caso de avioneta con droga en Manabí Trece personas involucradas en el caso de una avioneta encontrada con droga en Manabí fueron sentenciados a 17 años y 4 meses de prisión por un tribunal de Ecuador.",
    "text_b": "Fiscalía logra sentencia de 17 años 4 meses en contra de 13 personas involucradas en el caso de una avioneta que fue encontrada con más de media tonelada de droga, en diciembre de 2017, en el aeropuerto \"Los Perales\", en el cantón San Vicente, Manabí.",
    "similarity": 0.7812968492507935
  },
  {
    "doc_id": 2402,
    "class": "False",
    "type": "Redundante",
    "text_a": "En la convención sobre el Cambio Climático (UNFCC) con tema “una completa transformación de la estructura económica del mundo”, dijo en repetidas ocasiones que una dictadura comunista al estilo chino es la más adecuada para “calentamiento global”.",
    "text_b": "Pero cuál es la relación entre la dictadura y el cambio climático, cuando China es uno de los países con más contaminación del mundo.",
    "similarity": 0.7503963112831116
  },
  {
    "doc_id": 1430,
    "class": "False",
    "type": "Redundante",
    "text_a": "Un año más, los Reyes presidieron la cena de un certamen al que este año se presentaron más de cuatrocientos simios de todo el mundo.",
    "text_b": "A la fiesta asistieron más de quinientas personas y unos trescientos animales, entre simios y monos.",
    "similarity": 0.7840729355812073
  },
  {
    "doc_id": 1333,
    "class": "True",
    "type": "Redundante",
    "text_a": "Tendencias Hallan 145 ballenas muertas en una playa remota de Nueva Zelanda Unas 145 ballenas piloto murieron en un varamiento masivo en un punto remoto de una pequeña isla de Nueva Zelanda, anunciaron este lunes las autoridades.",
    "text_b": "Cientos de ballenas piloto han aparecido muertas en una playa de Nueva Zelanda.",
    "similarity": 0.8259779214859009
  },
  {
    "doc_id": 1992,
    "class": "False",
    "type": "No Redundante",
    "text_a": "Vaso desechable completa 59 lavadas En lo que se considera un nuevo récord personal para la señora Margarita de Laverde, un vaso plástico desechable ha sido usado y lavado ya más de 59 veces en su hogar de la ciudad de Ibagué.",
    "text_b": "Con esta justificación, la anciana recicla velas de cumpleaños, bolsas y papel regalo de celebraciones anteriores.",
    "similarity": 0.30181828141212463
  },
  {
    "doc_id": 1955,
    "class": "False",
    "type": "No Redundante",
    "text_a": "Él fue el encargado de llevar los avales a la sede de Ferraz, uno de los momentos icónicos en aquel proceso frente a Pilar Lima.",
    "text_b": "Cerdán atiende a Público en una entrevista sobre la expectativas electorales del Nueva Canarias, el papel del BNG de Ni Belarra, la situación a la izquierda de los socialistas y otras cuestiones de actualidad.",
    "similarity": 0.1744663119316101
  },
  {
    "doc_id": 1312,
    "class": "False",
    "type": "No Redundante",
    "text_a": "El Iniciativa vers per Catalunya no dudó y formó gobierno con EQUO.",
    "text_b": "Hoy el objetivo de la derecha es que el cambio en Andalucía no se quede en una anécdota, sino que se instale durante años",
    "similarity": 0.285478800535202
  },
  {
    "doc_id": 1446,
    "class": "True",
    "type": "No Redundante",
    "text_a": "El PSC avanza por todas partes y se beneficia del efecto Pedro Sánchez, mientras que la derecha española obtiene unos pésimos resultados, con un PP que fracasa y suma solo un diputado y un Cs que no mejora.",
    "text_b": "Vox obtiene un diputado en Barcelona.",
    "similarity": 0.34572139382362366
  },
  {
    "doc_id": 412,
    "class": "False",
    "type": "No Redundante",
    "text_a": "El permiso por maternidad le mantendrá alejada de la actividad política una temporada: se reincorporará de cara a la precampaña electoral de las municipales del 28 de mayo.",
    "text_b": "Por eso, aprovecha estas semanas con un ritmo de trabajo frenético, pero cuidándose, hasta que el cuerpo aguante.",
    "similarity": 0.1847328394651413
  }
]

for i, p in enumerate(RANDOM_PAIRS, 1):
    print(f'Par {i:02d} [{p["type"]} | Doc {p["doc_id"]} | Sim: {p["similarity"]:.4f}]:')
    print(f'  A: {p["text_a"]}')
    print(f'  B: {p["text_b"]}\n')


Pares aleatorios seleccionados: 10
Par 1 [Redundante]: Sim=0.7687
Par 2 [Redundante]: Sim=0.7813
Par 3 [Redundante]: Sim=0.7504
Par 4 [Redundante]: Sim=0.7841
Par 5 [Redundante]: Sim=0.8260
Par 6 [No Redundante]: Sim=0.3018
Par 7 [No Redundante]: Sim=0.1745
Par 8 [No Redundante]: Sim=0.2855
Par 9 [No Redundante]: Sim=0.3457
Par 10 [No Redundante]: Sim=0.1847


## 2. Carga del Modelo `paraphrase-multilingual-MiniLM-L12-v2`


In [4]:
MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True)
model.eval()

def mean_pool(h, mask):
    m = mask.unsqueeze(-1).expand(h.size()).float()
    return (h * m).sum(dim=1) / torch.clamp(m.sum(dim=1), min=1e-9)

def compute_similarity(text_a, text_b, curr_model=model):
    tok_a = tokenizer(text_a, return_tensors='pt', padding=True, truncation=True)
    tok_b = tokenizer(text_b, return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        out_a = curr_model(**tok_a)[0]
        out_b = curr_model(**tok_b)[0]
        vec_a = F.normalize(mean_pool(out_a, tok_a['attention_mask']), p=2, dim=1)
        vec_b = F.normalize(mean_pool(out_b, tok_b['attention_mask']), p=2, dim=1)
        return float((vec_a * vec_b).sum().item())
print('Modelo MiniLM-L12 cargado exitosamente en memoria.')


Modelo MiniLM-L12 cargado exitosamente en memoria.


## 3. Resumen Consolidado de Resultados Cuantitativos en Muestras Aleatorias


In [5]:
faith_summary = {
  "Saliency": {
    "comprehensiveness_mean": 0.18340017423033714,
    "sufficiency_mean": 0.21351512521505356
  },
  "IxG": {
    "comprehensiveness_mean": 0.1183264434337616,
    "sufficiency_mean": 0.29732863456010816
  },
  "Fast-IG": {
    "comprehensiveness_mean": 0.13455967903137206,
    "sufficiency_mean": 0.2329816535115242
  },
  "Attention": {
    "comprehensiveness_mean": 0.1473817393183708,
    "sufficiency_mean": 0.10114426165819168
  },
  "LIME-Light": {
    "comprehensiveness_mean": 0.3233832225203514,
    "sufficiency_mean": 0.11701572984457016
  },
  "SHAP-Light": {
    "comprehensiveness_mean": 0.16802409933879972,
    "sufficiency_mean": 0.24585577994585037
  }
}
latencies_summary = {
  "Saliency": 219.21115500445012,
  "IxG": 438.42231000890024,
  "Fast-IG": 3488.932829996338,
  "Attention": 104.51468000246678,
  "LIME-Light": 4875.121400010539,
  "SHAP-Light": 4506.167800005642
}
df_faith = pd.DataFrame(faith_summary).T
df_faith['latency_ms'] = pd.Series(latencies_summary)
display(df_faith)


=== RESUMEN CUANTITATIVO DE FIDELIDAD (MUESTRAS ALEATORIAS) ===
            comprehensiveness_mean  sufficiency_mean
Saliency                  0.183400          0.213515
IxG                       0.118326          0.297329
Fast-IG                   0.134560          0.232982
Attention                 0.147382          0.101144
LIME-Light                0.323383          0.117016
SHAP-Light                0.168024          0.245856

=== PERFIL DE LATENCIA PROMEDIO (ms) ===
Saliency       219.211155
IxG            438.422310
Fast-IG       3488.932830
Attention      104.514680
LIME-Light    4875.121400
SHAP-Light    4506.167800


## 4. Sanity Check de Adebayo (Aleatorización en Cascada)


In [6]:
sanity_results = [
  {
    "stage": "Original (0 Capas)",
    "randomized_layers_count": 0,
    "spearman_ixg": 1.0,
    "spearman_ig": 1.0
  },
  {
    "stage": "Capa 11 (Top-1)",
    "randomized_layers_count": 1,
    "spearman_ixg": 0.7872038409351844,
    "spearman_ig": 0.933451623003862
  },
  {
    "stage": "Capas 11-10 (Top-2)",
    "randomized_layers_count": 2,
    "spearman_ixg": 0.5200083498590962,
    "spearman_ig": 0.7662456946039037
  },
  {
    "stage": "Capas 11-8 (Top-4)",
    "randomized_layers_count": 4,
    "spearman_ixg": 0.19791253522596808,
    "spearman_ig": 0.7025780189959296
  },
  {
    "stage": "Capas 11-6 (Top-6)",
    "randomized_layers_count": 6,
    "spearman_ixg": 0.25176912639599214,
    "spearman_ig": 0.5899384197891661
  },
  {
    "stage": "Capas 11-0 (Todas)",
    "randomized_layers_count": 12,
    "spearman_ixg": 0.05959711929861185,
    "spearman_ig": 0.04949378979229726
  }
]
df_san = pd.DataFrame(sanity_results)
display(df_san)


              stage  randomized_layers_count  spearman_ixg  spearman_ig
 Original (0 Capas)                        0      1.000000     1.000000
    Capa 11 (Top-1)                        1      0.787204     0.933452
Capas 11-10 (Top-2)                        2      0.520008     0.766246
 Capas 11-8 (Top-4)                        4      0.197913     0.702578
 Capas 11-6 (Top-6)                        6      0.251769     0.589938
 Capas 11-0 (Todas)                       12      0.059597     0.049494
